In [39]:
!pip install kaggle xgboost shap imbalanced-learn gradio networkx

In [40]:
pip install optuna

In [41]:
import json

kaggle_info = {
    "username": "**********",
    "key": "**********"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_info, f)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d ealaxi/paysim1
!unzip paysim1.zip

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
paysim1.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  paysim1.zip
replace PS_20174392719_1491204439457_log.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd

df = pd.read_csv(
    "PS_20174392719_1491204439457_log.csv",
    nrows=100000
)

df.head()

In [ ]:
df['isFraud'].value_counts()

In [ ]:
import networkx as nx

G = nx.from_pandas_edgelist(
    df,
    'nameOrig',
    'nameDest',
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
print("Computing graph features...")

pagerank = nx.pagerank(G)
between = nx.betweenness_centrality(G, k=200)
clustering = nx.clustering(G.to_undirected())
df['orig_pagerank'] = df['nameOrig'].map(pagerank)
df['dest_degree'] = df['nameDest'].map(dict(G.in_degree()))
df['dest_pagerank'] = df['nameDest'].map(pagerank)
df['dest_between'] = df['nameDest'].map(between)
df['dest_cluster'] = df['nameDest'].map(clustering)


In [ ]:
df['velocity'] = df.groupby('nameOrig')['amount'].transform(lambda x: x.diff().fillna(0))

df['tx_count'] = df.groupby('nameOrig')['amount'].transform('count')
df['avg_tx_amount'] = df.groupby('nameOrig')['amount'].transform('mean')

df['amount_deviation'] = df['amount'] - df['avg_tx_amount']

In [ ]:
df['balance_error'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['dest_balance_diff'] = df['newbalanceDest'] - df['oldbalanceDest']

In [ ]:
df = pd.get_dummies(df, columns=['type'], drop_first=True)

In [ ]:
features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'dest_degree',
    'dest_pagerank',
    'orig_pagerank',
    'dest_between',
    'dest_cluster',
    'velocity',
    'tx_count',
    'avg_tx_amount',
    'amount_deviation',
    'balance_error',
    'dest_balance_diff'
] + [c for c in df.columns if c.startswith("type_")]

X = df[features]
y = df['isFraud']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE()

X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train.value_counts())

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score
def objective(trial):

    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 600)
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict_proba(X_test)[:,1]
    score = average_precision_score(y_test, preds)

    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=550,
    max_depth=5,
    learning_rate=0.012178711667131047,
    eval_metric='aucpr',
    tree_method='hist',
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, average_precision_score

y_probs = model.predict_proba(X_test)[:,1]
y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

print("AUPRC:", average_precision_score(y_test,y_probs))

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

target_recall = 0.85

idx = np.where(recalls >= target_recall)[0][-1]

FINAL_THRESHOLD = thresholds[idx]

print("Optimal threshold:", FINAL_THRESHOLD)

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

X_sample = X_test.sample(200)

shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample)

In [ ]:
fraud_df = df[df["isFraud"] == 1]

G_fraud = nx.from_pandas_edgelist(
    fraud_df,
    "nameOrig",
    "nameDest"
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))
nx.draw(G_fraud, node_size=50)
plt.title("Fraud Transaction Network")
plt.show()

In [ ]:
from xgboost import plot_importance

plot_importance(model, max_num_features=10)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

nn = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),

    layers.Dense(1, activation="sigmoid")
])

nn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

nn.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2
)

In [ ]:
import joblib

joblib.dump(model,"fraud_xgboost.pkl")

nn.save("fraud_nn.keras")

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

seq_data = df.sort_values("step")

user_sequences = seq_data.groupby("nameOrig")["amount"].apply(list)

X_seq = pad_sequences(user_sequences, maxlen=10)

In [ ]:
y_seq = seq_data.groupby("nameOrig")["isFraud"].max()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_model = Sequential([
    LSTM(64, input_shape=(10,1)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["AUC"]
)

lstm_model.fit(
    X_seq,
    y_seq,
    epochs=10,
    batch_size=256
)

In [ ]:
def predict_fraud(
    amount, old_balance, new_balance,
    old_dest_balance, new_dest_balance,
    dest_degree, velocity,
    type_transfer, type_cashout
):

    try:

        # -----------------------------
        # Feature Engineering
        # -----------------------------
        balance_error = old_balance - amount - new_balance
        dest_balance_diff = new_dest_balance - old_dest_balance

        input_dict = {
            "amount": amount,
            "oldbalanceOrg": old_balance,
            "newbalanceOrig": new_balance,
            "dest_degree": dest_degree,
            "dest_pagerank": 0.001,
            "orig_pagerank": 0.001,
            "dest_between": 0.0,
            "dest_cluster": 0.0,
            "velocity": velocity,
            "tx_count": 1,
            "avg_tx_amount": amount,
            "amount_deviation": 0,
            "balance_error": balance_error,
            "dest_balance_diff": dest_balance_diff
        }

        # -----------------------------
        # Transaction Type Encoding
        # -----------------------------
        for col in features:
            if col.startswith("type_"):
                input_dict[col] = 0

        if "type_TRANSFER" in features:
            input_dict["type_TRANSFER"] = int(type_transfer)

        if "type_CASH_OUT" in features:
            input_dict["type_CASH_OUT"] = int(type_cashout)

        # -----------------------------
        # Create Input DataFrame
        # -----------------------------
        input_df = pd.DataFrame([input_dict])

        # Ensure column order matches training
        input_df = input_df.reindex(columns=features, fill_value=0)

        # -----------------------------
        # Model Predictions
        # -----------------------------

        # XGBoost
        xgb_prob = float(model.predict_proba(input_df)[0][1])

        # Neural Network
        nn_prob = float(nn.predict(input_df.values, verbose=0)[0][0])

        # LSTM Sequence Model
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        import numpy as np

        seq = pad_sequences([[amount]], maxlen=10)
        seq = seq.reshape((1, 10, 1))

        lstm_prob = float(lstm_model.predict(seq, verbose=0)[0][0])

        # -----------------------------
        # Weighted Ensemble
        # -----------------------------
        weights = {
            "xgb": 0.5,
            "nn": 0.3,
            "lstm": 0.2
        }

        final_prob = (
            weights["xgb"] * xgb_prob +
            weights["nn"] * nn_prob +
            weights["lstm"] * lstm_prob
        )

        # -----------------------------
        # Decision Logic
        # -----------------------------
        if final_prob >= 0.85:
            status = "🚨 FRAUD DETECTED"
            risk = "HIGH RISK"

        elif final_prob >= 0.60:
            status = "⚠️ SUSPICIOUS"
            risk = "MEDIUM RISK"

        else:
            status = "✅ NORMAL"
            risk = "LOW RISK"

        # -----------------------------
        # Optional Debug Info
        # -----------------------------
        # print("XGB:", xgb_prob)
        # print("NN:", nn_prob)
        # print("LSTM:", lstm_prob)
        # print("FINAL:", final_prob)

        return status, f"{final_prob*100:.2f}%", risk

    except Exception as e:
        return "Error", str(e), "Check console"

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🛡 Sentinel AI v3")
    gr.Markdown("Graph + Behavioral + ML Fraud Detection")

    with gr.Row():

        with gr.Column():
            amt=gr.Number(label="Amount")
            oldb=gr.Number(label="Sender Old Balance")
            newb=gr.Number(label="Sender New Balance")
            oldd=gr.Number(label="Receiver Old Balance")
            newd=gr.Number(label="Receiver New Balance")
            deg=gr.Slider(0,100)
            vel=gr.Number(label="Velocity")
            t1=gr.Checkbox(label="Transfer")
            t2=gr.Checkbox(label="Cashout")
            btn=gr.Button("Analyze")

        with gr.Column():
            out1=gr.Textbox(label="Decision")
            out2=gr.Textbox(label="Fraud Probability")
            out3=gr.Textbox(label="Risk Level")

    btn.click(
        predict_fraud,
        inputs=[amt,oldb,newb,oldd,newd,deg,vel,t1,t2],
        outputs=[out1,out2,out3]
    )

demo.launch(share=True)